# 01 · Contrast prompt sets
Harmful prompts from Aya Red-teaming, harmless ones from the Aya Dataset, in English, French and Hindi. All logic is in `src/forged/data.py`; settings in `configs/contrast_sets.yaml`.

Run `00_setup` first (clone + install + HF login).

In [ ]:
# Clone the private repo and install the package (+ Lexsi libraries).
# Needs a GitHub token with read access stored as a Colab secret named GH_TOKEN.
import os, subprocess
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'mathcrypto/forged-fingerprints'
if not os.path.exists('/content/forged-fingerprints'):
    subprocess.run(['git', 'clone', f'https://{os.environ["GH_TOKEN"]}@github.com/{REPO}.git', '/content/forged-fingerprints'], check=True)
else:
    subprocess.run(['git', '-C', '/content/forged-fingerprints', 'pull'], check=True)
%cd /content/forged-fingerprints

## 1. Check the config and the Aya Dataset language names

In [ ]:
from forged.data import ContrastConfig, aya_language_counts
cfg = ContrastConfig.from_yaml('configs/contrast_sets.yaml')
counts = aya_language_counts()
for lang, lc in cfg.languages.items():
    print(lang, {n: counts.get(n, 0) for n in lc['aya_names']})

## 2. Build: load, screen, deduplicate, length-match, token check, split, hash

In [ ]:
from transformers import AutoTokenizer
from forged.data import build_contrast_sets
from forged.fingerprint import chat_token_counter
tok = AutoTokenizer.from_pretrained('CohereLabs/tiny-aya-global')
manifest = build_contrast_sets(cfg, token_counter=chat_token_counter(tok))
print(manifest['sizes'])
print('PROBE COMMITMENT (publish this):', manifest['probe_commitment_sha256'])

## 3. Read a sample
Remove anything borderline (e.g. lock picking) before pushing: clearly harmful vs. clearly harmless gives a cleaner fingerprint.

In [ ]:
import random
from forged.data import read_jsonl
rows = read_jsonl(cfg.out_dir / 'public' / 'A.jsonl')
for lang in cfg.languages:
    for label in ['harmful', 'harmless']:
        sub = [r['prompt'] for r in rows if r['language'] == lang and r['label'] == label]
        print(f'===== {lang} {label}')
        for p in random.Random(1).sample(sub, 5): print(' -', p[:160])

## 4. Push to Hugging Face (two private repos)

In [ ]:
HF_ORG = 'YOUR-HF-ORG'   # <- change me
!python scripts/build_contrast_sets.py --config configs/contrast_sets.yaml --push-org {HF_ORG}